In [0]:
#import requests
#symbols = "USD"
#base = "BRL"
#url = f"https://api.apilayer.com/exchangerates_data/latest?symbols={symbols}&base={base}"
#payload = {}
# Recupera a chave salva no Secret Scope
#api_key = dbutils.secrets.get(scope="projeto-cotacoes-moedas", key="Exchange-Rates-Data-API")
#headers= {
#  "apikey": f"{api_key}"
#}
#response = requests.request("GET", url, headers=headers, data = payload)
#status_code = response.status_code
#result = response.text

In [0]:
from pyspark.sql.functions import current_timestamp, col

# 1. Definição de caminhos e catálogo
caminho_json = spark.read.option("multiLine", "true").json("/Volumes/workspace/int_dados/moedas_cotacoes/Cotacoes_moedas.json")

# Criar/garantir o database/schema no catálogo exp-dados
spark.sql("CREATE DATABASE IF NOT EXISTS exp_dados")

# 2. Leitura do JSON bruto
df_raw = caminho_json

# 3. Adição de colunas de auditoria
df_bronze = df_raw.withColumn("_data_ingestao", current_timestamp()) \
                   .withColumn("_arquivo_origem", col("_metadata.file_path"))

# 4. Escrita na tabela Delta Bronze
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("exp_dados.bronze_cotacoes_raw")

print("Tabela Bronze criada com sucesso!")